# Lab 19 — Full Fine-Tuning vs LoRA Resource Tradeoff
**Goal:** Use optimizer-state accounting to compare the memory problem created by full fine-tuning and PEFT.

Free-first: CPU or free Colab where possible; expensive paths are clearly marked optional.



In [ ]:
def gib(x): return x/1024**3
def state_bytes(p,w=2,g=2,adam=8): return p*(w+g+adam)
for n in [135e6,360e6,500e6,1e9,7e9]:
    print(n/1e9,"B ->",gib(state_bytes(n)),"GiB before activations")


In [ ]:
def lora_adapter_params(in_dim,out_dim,r): return r*(in_dim+out_dim)
for r in [8,16,32]:
    print("rank",r,"for 4096x4096:",lora_adapter_params(4096,4096,r),"params")


In [ ]:
# Experiment design template
print({
 "fixed":["base model","dataset","evaluation","sequence length","training budget"],
 "changed":["parameter update strategy"],
 "outputs":["quality","memory","throughput","storage","failure modes"]
})


### Answer
Full fine-tuning updates all model parameters and therefore carries their gradients and optimizer states (plus activations). LoRA trains only a small set of additional parameters. The correct choice depends on objective, data size, domain shift, quality target, memory budget, and operational goals.


## Primary source
https://huggingface.co/docs/transformers/main/trainer


## Deepening: compare adaptation methods fairly

This section turns the notebook into a quantitative lab. Predict before executing.


In [ ]:
N = 7_000_000_000
bf16_bytes = 2 * N
adapter_rank = 16
d_in = d_out = 4096
lora_per_matrix = adapter_rank * (d_in + d_out)
full_per_matrix = d_in * d_out

print("7B raw BF16 weights GB:", bf16_bytes/1e9)
print("example full matrix params:", full_per_matrix)
print("example LoRA params:", lora_per_matrix)
print("matrix parameter ratio:", lora_per_matrix/full_per_matrix)


## Break experiment

Use the same model, data, evaluation set, and approximate training-token budget.

Compare:

**LoRA → QLoRA → full FT**

Report target quality, retained capability, peak memory, wall time, checkpoint size, and cost.

The question is not “which method is best?” but “what trade-off did each intervention produce under the same constraints?”

Complete templates/EXPERIMENT_CARD.md for the final result.
